In [48]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import collections
import itertools
import tqdm
from copy import deepcopy
from functools import lru_cache

In [49]:
np.random.seed(0)

In [50]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [51]:
def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities, axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [52]:
def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [53]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    acc_loss_p = accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p
    return acc_loss_p

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [54]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [55]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))

    @lru_cache(maxsize=None)
    def evaluate_partition_cached(partition_tuple):
        partition = list(partition_tuple)
        acc_loss_p = evaluate_partition(
            X, partition, thresholds, priors, threshold_true, c
        )
        return acc_loss_p * np.sum(priors[partition])

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.0
        for partition in partitions:
            partition_tuple = tuple(sorted(partition))
            acc_loss += evaluate_partition_cached(partition_tuple)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [56]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)

def find_partitions_greedy(X, thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1


    Q = collections.deque(itertools.combinations(P.keys(), 2))

    while Q:
        if display:
            display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-6:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for c_id in P.keys():
                if c_id != new_id:
                    Q.append((new_id, c_id))

            # Q = collections.deque(sorted(Q, key=lambda x: P[x[0]]))
    return list(P.values())

In [57]:
def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if (1 - loss_greedy) == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [58]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)
N_REPS = 100
N_THRESHOLDS = 3

In [59]:
def generate_prior_grid(n_components=3, n_balls=50):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls):
        counts = np.bincount(combo, minlength=n_components)
        grids.append(counts * step)
    return grids

In [ ]:
def generate_threshold_grid(n_components=3, step=0.02):
    ticks = np.round(np.arange(0, 1 + step, step), 10)
    return [np.array(combo) for combo in itertools.combinations(ticks, n_components)]

In [ ]:
np.random.seed(0)

def sweep(threshold_grid, prior_grid):
    d = {"thresholds": [], "priors": [], "c": [], "t*": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r_mult": [], "r_add": []}
    c = 0.5
    # threshold_true = np.min(thresholds)
    # threshold_true = np.median(thresholds)
    # threshold_true = 0.5
    for thresholds, priors in tqdm.tqdm(itertools.product(threshold_grid, prior_grid), total=len(threshold_grid)*len(prior_grid)):
        threshold_true = thresholds[0]
        partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
        partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
        r_mult  = approximation_ratio(loss_opt, loss_greedy, rtype="mult")
        r_add  = approximation_ratio(loss_opt, loss_greedy, rtype="add")
        if not np.isnan(r_mult):
            d["thresholds"].append(thresholds)
            d["priors"].append(priors)
            d["c"].append(c)
            d["t*"].append(threshold_true)
            d["partition_opt"].append(partition_opt)
            d["partition_greedy"].append(partition_greedy)
            d["loss_opt"].append(loss_opt.item())
            d["loss_greedy"].append(loss_greedy.item())
            d["r_mult"].append(r_mult.item())
            d["r_add"].append(r_add.item())

    return d

In [62]:
N_THRESHOLDS = 3
threshold_grid = generate_threshold_grid(n_components=N_THRESHOLDS, step=0.1)
prior_grid = generate_prior_grid(N_THRESHOLDS, n_balls=100)
d = sweep(threshold_grid, prior_grid)

100%|██████████| 849915/849915 [1:31:02<00:00, 155.58it/s]


In [63]:
df = pd.DataFrame(d)
i = np.argmax(d["r_mult"])
df.iloc[[i]]

,thresholds,priors,c,t*,partition_opt,partition_greedy,loss_opt,loss_greedy,r_mult,r_add
762778,"[0.5, 0.6, 1.0]","[0.72, 0.04, 0.24]",0.5,0.5,"[[1], [0, 2]]","[[0, 1, 2]]",0.38121,0.49995,1.311482,0.11874


In [67]:
thresholds = d["thresholds"][i]
priors = d["priors"][i]
threshold_true = 0.48
c = 1.

partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
# partition_greedy = d["partition_greedy"][i]
partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)

loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
# loss_greedy = d_tt["loss_greedy"][i]
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)

r = approximation_ratio(loss_opt, loss_greedy)

In [68]:
r

np.float64(1.0)